In [1]:
import pandas as pd

In [32]:
file_aligment = r'../gold/aligment_all.csv'
file_aligment_analyst = r'../gold/aligment_all_analyst.csv'
df = pd.read_csv(file_aligment, encoding='utf-8')

In [33]:
df.head()

,акт,обьект,по_Y,по_X,процент_нечеткого_поиска
0,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,Unit No: 1 Date of inspection: 2022-12-07,32.258065
1,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,Incoming Inspection Certificate of Materials o...,80.645161
2,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,Pages from 1 to 1 with Attachments on 78 pages,25.806452
3,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,INCOMING INSPECTION CERTIFICATE OF MATERIALS O...,19.672131
4,ED.A009.MII.&.012.0028,number_act,ED.A009.MII.&.012.0028,Manufacturer: LLC AEMZ; OMEGA METALL LLC; EVRA...,25.806452


In [34]:
def one_max(df, name_column, diff=0):
    '''добавляет столбец по одному максиммуму на акт'''
    need_column = (df['обьект'] == name_column) & (~df['по_Y'].isna())
    # idx_max = df[need_column].groupby(['акт'])['процент_нечеткого_поиска'].idxmax()
    max_val = df.where(need_column).groupby('акт')['процент_нечеткого_поиска'].transform('max')
    df[name_column] = 0
    is_max = need_column & (df['процент_нечеткого_поиска'] >= (max_val - diff))
    df.loc[is_max, name_column] = 1

def n_max(df, name_column, diff=0):
    '''добавляет столбец по одному максиммуму на акт и на Y'''
    need_column = (df['обьект'] == name_column) & (~df['по_Y'].isna())
    max_val = df.where(need_column).groupby(['акт', 'по_Y'])['процент_нечеткого_поиска'].transform('max')
    df[name_column] = 0
    is_max = need_column & (df['процент_нечеткого_поиска'] >= (max_val - diff))
    df.loc[is_max, name_column] = 1

In [35]:
df['обьект'].unique()

array(['number_act', 'block', 'commission', 'date', 'conclusion',
       'material___name', 'material___batch',
       'material___quantity___value', 'material___quantity___unit',
       'manufacturer', 'documents', 'invoices', 'projects'], dtype=object)

In [36]:
# добавляем отметки
one_max(df, 'number_act')
one_max(df, 'block')
one_max(df, 'conclusion')
one_max(df, 'date')
n_max(df, 'manufacturer')
n_max(df, 'commission')
n_max(df, 'invoices')
n_max(df, 'projects')
n_max(df, 'documents')

In [37]:
df['сумма'] = df[df.columns[5:]].sum(axis=1)

In [38]:
# удалить из обучения акты  #акт не соответствует, проверял вручную
need_del = [
    'ED.A009.MII.&.012.0381',
    'ED.A009.MII.&.012.0401',
    'ED.A009.MII.&.012.0285',
    'ED.A023.MII.&.&&&.01299-II',
    'ED.A009.MII.&.012.0442',
    'ED.A009.MII.&.012.0143',
    'ED.A009.MII.&.012.1007',
    'ED.A023.MII.&.&&&.01931-II',
    'ED.A009.MII.&.015.0997',
    'ED.A009.MII.&.012.0368',
    'ED.A009.MII.&.012.1007'
]
df = df[~df['акт'].isin(need_del)]

In [39]:
# кол-во актов
df['акт'].nunique()

884

In [40]:
# пометить нулями где процент_нечеткого_поиска меньше 70 процентов (процент на глаз вручную)
mask = (df['сумма'] >= 1) & (df['процент_нечеткого_поиска'] < 70.0)
df.loc[mask, 'сумма'] = 0

In [41]:
# пометить not_group
mask = df['сумма'] == 0
df.loc[mask, 'обьект'] = 'not_group'
df.loc[mask, 'по_Y'] = pd.NA

In [42]:
# отсортировать дубликаты

# ранг имени ВНУТРИ каждого акт (порядок первого появления в этом id)
df['_x_rank'] = df.groupby('акт', sort=False)['по_X'].transform(
    lambda s: s.map({n: i for i, n in enumerate(s.drop_duplicates())})
)

# ранг id по порядку первого появления (чтобы id шли блоками в исходном порядке)
id_order = {i: k for k, i in enumerate(df['акт'].drop_duplicates())}
df['_act_rank'] = df['акт'].map(id_order)

df = (df
      .sort_values(['_act_rank', '_x_rank', 'сумма'],
                   ascending=[True, True, False], kind='stable')
      .drop(columns=['_x_rank', '_act_rank'])
      .reset_index(drop=True))


In [43]:
# удалить дубликаты из актов по X в not_group

def filter_group(g):
    ones = g[g['сумма'] >= 1]
    if len(ones) > 0:
        return ones          # есть единицы → оставляем только их
    return g.head(1)         # все нули → только первая строка

df = (df
      .groupby(['акт', 'по_X'], sort=False, group_keys=False)
      .apply(filter_group)
      .reset_index(drop=True))

df = df.drop_duplicates(keep='first')

C:\Temp\ipykernel_5920\2033734838.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(filter_group)


In [44]:
df.to_csv(file_aligment_analyst, index=False, encoding='utf-8-sig')

In [18]:
df = pd.DataFrame({
    'id':   [3, 3, 3, 3,3, 2, 2],
    'name': ['A', 'B', 'A', 'A', 'B', 'C', 'A'],
    'val':  [0, 0, 1, 0, 0, 1, 0],
    'test': [1,2,3,4,5,6,7]
})

# ранг имени ВНУТРИ каждого id (порядок первого появления в этом id)
df['_name_rank'] = df.groupby('id', sort=False)['name'].transform(
    lambda s: s.map({n: i for i, n in enumerate(s.drop_duplicates())})
)

# ранг id по порядку первого появления (чтобы id шли блоками в исходном порядке)
id_order = {i: k for k, i in enumerate(df['id'].drop_duplicates())}
df['_id_rank'] = df['id'].map(id_order)

df = (df
      .sort_values(['_id_rank', '_name_rank', 'val'],
                   ascending=[True, True, False], kind='stable')
      .drop(columns=['_name_rank', '_id_rank'])
      .reset_index(drop=True))

print(df)

def filter_group(g):
    ones = g[g['val'] == 1]
    if len(ones) > 0:
        return ones          # есть единицы → оставляем только их
    return g.head(1)         # все нули → только первая строка

df = (df
      .groupby(['id', 'name'], sort=False, group_keys=False)
      .apply(filter_group)
      .reset_index(drop=True))

print(df)

   id name  val  test
0   3    A    1     3
1   3    A    0     1
2   3    A    0     4
3   3    B    0     2
4   3    B    0     5
5   2    C    1     6
6   2    A    0     7
   id name  val  test
0   3    A    1     3
1   3    B    0     2
2   2    C    1     6
3   2    A    0     7


C:\Temp\ipykernel_5920\2250583558.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(filter_group)


In [45]:
df['name'].drop_duplicates()

0    A
2    B
5    C
Name: name, dtype: object

In [46]:
name_order

{'A': 0, 'B': 1, 'C': 2}

In [76]:
need_column = (df['обьект'] == 'manufacturer') & (~df['по_Y'].isna())
print(df.where(need_column).groupby('акт')['процент_нечеткого_поиска'].transform('max').mean())
print(df.where(need_column).groupby(['акт', 'по_Y'])['процент_нечеткого_поиска'].transform('max').mean())

86.8415057244308
83.47877101492378


In [40]:
df[(df['обьект'] == 'block') & (df['по_Y'].isna())]['block'].sum()

np.int64(0)